In [31]:
import numpy as np
import pandas as pd

In [32]:
df = pd.read_csv('../Datasets/machine_failure_dataset.csv')
df.head()

,Temperature,Vibration,Power_Usage,Humidity,Machine_Type,Failure_Risk
0,74.967142,56.996777,8.649643,20.460962,Mill,1
1,68.617357,54.623168,9.710963,25.698075,Lathe,0
2,76.476885,50.298152,8.415160,27.931972,Drill,1
3,85.230299,46.765316,9.384077,39.438438,Lathe,1
4,67.658466,53.491117,6.212771,32.782766,Drill,1


In [33]:
# One-Hot Encoding 
df = pd.get_dummies(df, columns=['Machine_Type'])
df.head()

,Temperature,Vibration,Power_Usage,Humidity,Failure_Risk,Machine_Type_Drill,Machine_Type_Lathe,Machine_Type_Mill
0,74.967142,56.996777,8.649643,20.460962,1,False,False,True
1,68.617357,54.623168,9.710963,25.698075,0,False,True,False
2,76.476885,50.298152,8.415160,27.931972,1,True,False,False
3,85.230299,46.765316,9.384077,39.438438,1,False,True,False
4,67.658466,53.491117,6.212771,32.782766,1,True,False,False


In [34]:
X = df.drop('Failure_Risk', axis=1).values
y = df['Failure_Risk'].values.reshape(-1, 1)
print(X.shape, y.shape)

(1000, 7) (1000, 1)


In [35]:
X = X.astype(float)

In [36]:
## Z-score estandarización o normalización
m = np.mean(X, axis=0)
std = np.std(X, axis=0)
X = (X - m) / (std + 1e-8)
# X[:5]

In [37]:
# inicializar parametros de la MLP
input_dim = X.shape[1]
hidden_dim = 8
output_dim = 1
lr = 0.05

In [38]:
# Inicialización de pesos y sesgos
np.random.seed(42)
W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

In [39]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [40]:
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    # La derivada es 1 si el valor es mayor a 0, de lo contrario es 0
    return (x > 0).astype(float)

In [41]:
# Entrenamiento de la red MLP
print("Entrenando...")
for epoch in range(1000):
    # Forward Pass
    z1 = np.dot(X, W1) + b1 # suma ponderada de la capa oculta
    a1 = relu(z1) # ReLU en la capa oculta
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)
    
    # Cálculo del error (Loss simple)
    error = a2 - y
    
    # Backpropagation
    dz2 = error * (a2 * (1 - a2)) # Derivada de la Sigmoide (salida)
    dW2 = np.dot(a1.T, dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)
    
    dz1 = np.dot(dz2, W2.T) * relu_derivada(a1) # Derivada de ReLU (capa oculta)
    dW1 = np.dot(X.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)
    
    # Actualización (Gradiente Descendente)
    W2 -= lr * (dW2 / len(X))
    b2 -= lr * (db2 / len(X))
    W1 -= lr * (dW1 / len(X))
    b1 -= lr * (db1 / len(X))

    # if epoch % 1000 == 0:
    loss = np.mean(np.square(error))
    print(f"Epoch {epoch} - Error (loss): {loss:.4f}")

Entrenando...
Epoch 0 - Error (loss): 0.2472
Epoch 1 - Error (loss): 0.2470
Epoch 2 - Error (loss): 0.2467
Epoch 3 - Error (loss): 0.2465
Epoch 4 - Error (loss): 0.2462
Epoch 5 - Error (loss): 0.2460
Epoch 6 - Error (loss): 0.2457
Epoch 7 - Error (loss): 0.2455
Epoch 8 - Error (loss): 0.2453
Epoch 9 - Error (loss): 0.2450
Epoch 10 - Error (loss): 0.2448
Epoch 11 - Error (loss): 0.2445
Epoch 12 - Error (loss): 0.2443
Epoch 13 - Error (loss): 0.2441
Epoch 14 - Error (loss): 0.2438
Epoch 15 - Error (loss): 0.2436
Epoch 16 - Error (loss): 0.2434
Epoch 17 - Error (loss): 0.2432
Epoch 18 - Error (loss): 0.2429
Epoch 19 - Error (loss): 0.2427
Epoch 20 - Error (loss): 0.2425
Epoch 21 - Error (loss): 0.2423
Epoch 22 - Error (loss): 0.2421
Epoch 23 - Error (loss): 0.2418
Epoch 24 - Error (loss): 0.2416
Epoch 25 - Error (loss): 0.2414
Epoch 26 - Error (loss): 0.2412
Epoch 27 - Error (loss): 0.2410
Epoch 28 - Error (loss): 0.2408
Epoch 29 - Error (loss): 0.2406
Epoch 30 - Error (loss): 0.2404
Epoc

In [42]:
predicciones = (a2 > 0.5).astype(int)
accuracy = np.mean(predicciones == y)

print(f"Precisión (Accuracy) sobre el total de datos: {accuracy * 100:.2f}%")

Precisión (Accuracy) sobre el total de datos: 70.00%
